# Módulo 2 — Introdução ao Pandas

**Pandas** é a biblioteca mais utilizada para análise de dados em Python. Ela fornece o `DataFrame` — uma tabela bidimensional com rótulos — e centenas de operações otimizadas.

## O que vamos ver

1. Importar pandas e criar DataFrames
2. Ler o CSV de consumidores simulados
3. Exploração básica: `.head()`, `.info()`, `.describe()`, `.shape`
4. Seleção de colunas e filtragem de linhas
5. Agrupamentos com `groupby`

---

> **Pré-requisito:** Certifique-se de estar com o ambiente `curso-python-energia` ativado e ter executado `jupyter lab` a partir do diretório raiz do curso.

In [ ]:
import pandas as pd
import numpy as np

# Verificar versões
print(f"Pandas: {pd.__version__}")
print(f"NumPy:  {np.__version__}")

## 1. Criando DataFrames Manualmente

Um DataFrame pode ser criado a partir de um dicionário, onde cada chave é uma coluna.

In [ ]:
# Criando um pequeno DataFrame de exemplo
dados = {
    "cod_consumidor": [100001, 100002, 100003, 100004],
    "nom_consumidor": ["Maria Silva", "Mercado Central", "João Oliveira", "Fábrica Alfa"],
    "cod_classe": ["RESIDENCIAL", "COMERCIAL", "RESIDENCIAL", "INDUSTRIAL"],
    "vlr_consumo_kwh": [245.3, 8920.5, 89.3, 125000.0],
    "flg_ativo": [True, True, True, True]
}

df_exemplo = pd.DataFrame(dados)
print("DataFrame criado:")
df_exemplo

## 2. Lendo o CSV de Consumidores

Na prática, os dados vêm de arquivos CSV, bancos de dados ou APIs.

In [ ]:
# Caminho relativo ao notebook (dentro de modulo2_dados_com_pandas/)
df = pd.read_csv("dados/consumidores_simulado.csv")

print(f"DataFrame carregado com sucesso!")
print(f"Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(f"Colunas: {list(df.columns)}")

## 3. Exploração Básica

In [ ]:
# Primeiras 5 linhas
df.head()

In [ ]:
# Últimas 3 linhas
df.tail(3)

In [ ]:
# Informações sobre tipos e valores nulos
df.info()

In [ ]:
# Estatísticas descritivas das colunas numéricas
df.describe()

In [ ]:
# Contagem de valores nulos por coluna
nulos = df.isnull().sum()
pct_nulos = (df.isnull().sum() / len(df) * 100).round(1)

resumo_nulos = pd.DataFrame({
    "nulos": nulos,
    "pct_nulos": pct_nulos
}).sort_values("nulos", ascending=False)

print("Valores nulos por coluna:")
resumo_nulos[resumo_nulos["nulos"] > 0]

## 4. Seleção de Colunas e Filtragem

In [ ]:
# Selecionar uma coluna (retorna Series)
consumos = df["vlr_consumo_medio_kwh"]
print(f"Tipo: {type(consumos)}")
print(f"Média: {consumos.mean():.1f} kWh")
print(f"Mediana: {consumos.median():.1f} kWh")

In [ ]:
# Selecionar múltiplas colunas (retorna DataFrame)
colunas_cadastro = ["cod_consumidor", "nom_consumidor", "cod_classe_consumo", "vlr_consumo_medio_kwh"]
df_resumo = df[colunas_cadastro]
df_resumo.head(5)

In [ ]:
# Filtrar apenas consumidores ativos
df_ativos = df[df["flg_ativo"] == True]
print(f"Ativos: {len(df_ativos)} de {len(df)}")

# Filtrar por classe
df_residenciais = df[df["cod_classe_consumo"] == "RESIDENCIAL"]
print(f"Residenciais: {len(df_residenciais)}")

# Filtro combinado: industriais com consumo > 50.000 kWh
df_grandes_industriais = df[
    (df["cod_classe_consumo"] == "INDUSTRIAL") &
    (df["vlr_consumo_medio_kwh"] > 50000)
]
print(f"\nGrandes industriais (>50.000 kWh):")
df_grandes_industriais[["cod_consumidor", "nom_consumidor", "vlr_consumo_medio_kwh"]]

In [ ]:
# Usando .query() — sintaxe mais legível
df_sp_ativos = df.query("cod_uf == 'SP' and flg_ativo == True")
print(f"Consumidores ativos em SP: {len(df_sp_ativos)}")

## 5. Agrupamentos com `groupby`

O `groupby` é equivalente ao `GROUP BY` do SQL — agrupa linhas e aplica funções de agregação.

In [ ]:
# Contagem de consumidores por classe
por_classe = df.groupby("cod_classe_consumo")["cod_consumidor"].count()
por_classe.name = "quantidade"
print("Consumidores por classe:")
print(por_classe.sort_values(ascending=False))

In [ ]:
# Múltiplas agregações
resumo_classe = df.groupby("cod_classe_consumo")["vlr_consumo_medio_kwh"].agg(
    qtd="count",
    media="mean",
    mediana="median",
    total="sum",
    maximo="max"
).round(1)

print("Resumo de consumo por classe:")
resumo_classe.sort_values("total", ascending=False)

In [ ]:
# Agrupamento por modalidade tarifária
por_modalidade = (
    df.groupby("cod_modalidade_tarifaria")
    .agg(
        qtd=("cod_consumidor", "count"),
        consumo_medio_kwh=("vlr_consumo_medio_kwh", "mean"),
        consumo_total_kwh=("vlr_consumo_medio_kwh", "sum")
    )
    .round(1)
    .sort_values("qtd", ascending=False)
)

print("Resumo por modalidade tarifária:")
por_modalidade

In [ ]:
# Percentual do consumo total por classe
consumo_total = df["vlr_consumo_medio_kwh"].sum()
participacao = (
    df.groupby("cod_classe_consumo")["vlr_consumo_medio_kwh"]
    .sum()
    .sort_values(ascending=False)
)

print(f"Consumo total: {consumo_total:,.0f} kWh\n")
print("Participação por classe:")
for classe, consumo in participacao.items():
    pct = consumo / consumo_total * 100
    barra = "█" * int(pct / 2)
    print(f"  {classe:<20}: {pct:>5.1f}% {barra}")

## 6. Ordenação e Seleção por Posição

In [ ]:
# Top 5 maiores consumidores
top5 = (
    df[["cod_consumidor", "nom_consumidor", "cod_classe_consumo", "vlr_consumo_medio_kwh"]]
    .sort_values("vlr_consumo_medio_kwh", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

top5.index = top5.index + 1  # índice começando em 1
print("Top 5 maiores consumidores:")
top5

In [ ]:
# Selecionar por índice com .iloc e .loc
# .iloc — posição numérica
print("Primeiro registro (iloc):")
print(df.iloc[0][["cod_consumidor", "nom_consumidor", "vlr_consumo_medio_kwh"]])

# .loc — rótulo do índice
print("\nRegistro com índice 4 (loc):")
print(df.loc[4, ["cod_consumidor", "nom_consumidor", "vlr_consumo_medio_kwh"]])